# wandb-watch-model — worked example 3: Watch a model and verify log_freq is passed correctly

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-watch-model`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `log_freq` parameter to `wandb.watch` controls the cadence of histogram logging — specifically, how many optimizer step calls happen between each histogram snapshot. A smaller value gives more granular histograms but increases wandb storage usage. The value is passed as a keyword argument and must match the intended logging cadence.

## Worked solution

**Step 1 — attach the watch.**
We call `wandb.watch(model, log='all', log_freq=log_freq)`. The `log_freq` is forwarded directly from the caller — this function doesn't hardcode a value, making it reusable across experiments with different logging granularity.

**Step 2 — return the log_freq for confirmation.**
We return the `log_freq` value passed to the function. The caller (or test) can compare this against `wandb.watch.call_args.kwargs['log_freq']` to verify the value was forwarded correctly.

**Step 3 — what log_freq means in practice.**
With `log_freq=10`, wandb logs histograms after every 10th call to `optimizer.step()`. If your training loop logs other metrics every step, the histogram cadence may look coarser on the dashboard — that's expected.

In [ ]:
import sys
from unittest.mock import MagicMock
import torch.nn as nn
sys.modules.setdefault('wandb', MagicMock())
import wandb

def attach_watch_with_freq(model, log_freq):
    """Attach wandb.watch to the entire model and return the log_freq."""
    wandb.watch(model, log='all', log_freq=log_freq)
    return log_freq

# Exercise it
wandb.watch.reset_mock()
model = nn.Linear(10, 2)
for freq in [10, 50, 200]:
    wandb.watch.reset_mock()
    returned = attach_watch_with_freq(model, freq)
    call_freq = wandb.watch.call_args.kwargs['log_freq']
    print(f'log_freq={freq}: returned={returned}, call_arg={call_freq}, match={returned == call_freq}')